In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
GOLD_FATO_PATH   = "workspace.case_spark_cvm.gold_fato_diario"
NOME_TABELA      = f"gold_cubo_rentabilidade" 
GOLD_PATH        = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC        = int(datetime.now().strftime("%Y%m%d"))

### 1. gold_fato_diario

In [0]:
gold_fato_diario  = spark.read.table(GOLD_FATO_PATH)

### 2. Criar Coluna Mensal

In [0]:
df_mensal = gold_fato_diario.withColumn(
  "ano_mes",
  f.date_trunc("month", f.col("dt_comptc"))
  
)

In [0]:
window_24m = Window.partitionBy("cnpj_fundo_classe").orderBy("ano_mes").rowsBetween(-23, 0)

df_mensal = gold_fato_diario \
    .groupBy(
        "cnpj_fundo_classe",
        "ano_mes"
    ) \
    .agg(
        # min_by: "Me dê o vl_quota do dia em que a dt_comptc for a menor do mês"
        f.expr("min_by(vl_quota, dt_comptc)").alias("quota_inicio_mes"),
        
        # max_by: "Me dê o vl_quota do dia em que a dt_comptc for a maior do mês"
        f.expr("max_by(vl_quota, dt_comptc)").alias("quota_fim_mes")
    ) \
    .withColumn( 
        "retorno_mensal",
        f.try_divide(f.col("quota_fim_mes"), f.col("quota_inicio_mes")) - 1
    ) \
    .withColumn(
        "mes_positivo",
        f.when(f.col("retorno_mensal") > 0, 1).otherwise(0)
    ) \
    .withColumn(
        "consistencia_retorno",
        f.avg("mes_positivo").over(window_24m)
    )

In [0]:
window_last = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("ano_mes").desc())

gold_consistencia = df_mensal\
.withColumn(
    "rn",
    f.row_number().over(window_last)
)\
.filter(f.col("rn") == 1)\
.select("cnpj_fundo_classe", "consistencia_retorno")

### 3. Pegando a Ultima data do Fundo

In [0]:
window_ultima_data = Window.partitionBy("cnpj_fundo_classe")

gold_fato_diario = gold_fato_diario\
    .withColumn(
        "ultima_dt_fundo",
        f.max(f.col("dt_comptc")).over(window_ultima_data)
)


### 4. Join 

In [0]:
gold_fato_diario = gold_fato_diario\
    .join(
        gold_consistencia,
        "cnpj_fundo_classe",
        "left"
    )

### 5. Selecionando as Colunas de Rentabilidade

In [0]:
gold_cubo_rentabilidade = gold_fato_diario\
    .filter(f.col("dt_comptc") == f.col("ultima_dt_fundo"))\
    .select(
        "cnpj_fundo_classe",
        f.col("dt_comptc").alias("dt_referencia"),
        "retorno_diario",
        "retorno_21d",
        "retorno_63d",
        "retorno_126d",
        "retorno_252d",
        "retorno_inicio",
        "retorno_benchmark_21d",
        "retorno_benchmark_63d",
        "retorno_benchmark_126d",
        "retorno_benchmark_252d",
        "alpha_21d",
        "alpha_63d",
        "alpha_126d",
        "alpha_252d",
        "benchmark_normalizado",
        "consistencia_retorno"
    )

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}") 

PipelineConfig.gravar_cubo_gold(
    spark=spark,
    df_cubo=gold_cubo_rentabilidade,
    tabela_destino=GOLD_PATH,
    zorder_cols=["cnpj_fundo_classe"]
)


log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")

In [0]:
%sql
select 
    *
from workspace.case_spark_cvm.gold_cubo_rentabilidade